naomi- added part of the code that will display the centroid image of each cluster for both pixel and clip space.

 

## Displaying the mean image of each cluster in both spaces

In [1]:
#Using starter code to set up StableUnCLIP in this notebook
#the same way as in the PCAperturbation notebooks

In [2]:
import requests
import torch
print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
from PIL import Image
import matplotlib.pyplot as plt
from io import BytesIO
import numpy as np
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
import matplotlib.gridspec as gridspec

from diffusers import StableUnCLIPImg2ImgPipeline
from transformers import CLIPTextModelWithProjection, CLIPTokenizer

Memory allocated: 0.00 GB


2025-12-09 21:57:24.816240: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-09 21:57:24.816288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-09 21:57:24.817592: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-09 21:57:24.824930: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
tqdm._instances.clear()

In [7]:
# Code for when GPU throws OOM error
torch.cuda.empty_cache()

try:
    del pipe
except:
    pass

try:
    del recon_pipe
except:
    pass

# Check whats left -- should see 0
print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

Memory allocated: 0.00 GB
Memory reserved: 0.00 GB


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
recon_pipe = StableUnCLIPImg2ImgPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-1-unclip", torch_dtype=torch.float32
)

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

In [10]:
def reconstruct_image_from_embedding(recon_pipe, embedding):
    emb_tensor = torch.from_numpy(embedding).unsqueeze(0).to(
        device=recon_pipe.device, 
        dtype=recon_pipe.image_encoder.dtype
    )
    return recon_pipe(image_embeds=emb_tensor).images[0]

In [11]:
# a function to visualize centroids
def show_all_centroid_images(centroids, cols, mode = 'auto'):
   
    #auto mode will use the dimensions of the centroids to determine whether it is pixel or CLIP
    if mode == 'auto':
        d = centroids.shape[1]
        if d == 1024:
            mode = 'CLIP'
        elif d == 49152:
            mode = 'pixel'
        else:
            raise ValueError (f"Cannot auto-detect mode for dimension {d}")
            
    num = len(centroids)
    rows = (num + cols - 1) // cols

    plt.figure(figsize = (cols * 3, rows * 3))
    
    for i, c in enumerate(centroids):

        #CLIP centroid image reconstruction
        if mode == 'CLIP':
            img = reconstruct_image_from_embedding(recon_pipe, c)

        #pixel centroid image reconstruction
        elif mode == 'pixel':
            img = c.reshape(128,128,3)
            img = np.clip(img, 0, 255).astype(np.uint8)
            img = Image.fromarray(img)
        plt.subplot(rows,cols, i + 1)
        plt.imshow(img)
        plt.title(f'C{i}')
        plt.axis('off')

    plt.suptitle('Mean images of all 45 CLIP clusters')
    plt.tight_layout()
    plt.show()

In [ ]:
#CLIP space
clipcentroids = kmeans_clip.cluster_centers_
show_all_centroid_images(clipcentroids, cols = 8)

In [ ]:
#Pixel space
pixelcentroids = pca_pixels.inverse_transform(kmeans_pixel.cluster_centers_)
show_all_centroid_images(pixelcentroids, cols = 8)
#show_all_centroid_images(pixelcentroids, cols = 8)